# Préparation des données — Candidats au classement

Ce notebook extrait et nettoie les colonnes depuis `candidat_classement.csv`.
Seul `Niveau_Math` est standardisé (variantes identiques → 2 labels propres).
`Diplome` est conservé tel quel après nettoyage des espaces/encodage.

In [1]:
import pandas as pd
import numpy as np

for enc in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
    try:
        df_raw = pd.read_csv('candidat_classement.csv', encoding=enc, sep=',')
        print(f'Encodage : {enc}')
        break
    except UnicodeDecodeError:
        continue

print(f'{df_raw.shape[0]} candidats, {df_raw.shape[1]} colonnes')

Encodage : utf-8
815 candidats, 134 colonnes


In [2]:
# Identifier la colonne Diplôme quelle que soit l'orthographe / l'encodage
diplome_col = next((c for c in df_raw.columns if c.lower().replace('\xa0','').startswith('dipl')), None)

COLONNES = {
    'ID'                                          : 'ID',
    diplome_col                                   : 'Diplome',
    'Moyenne L1, L2 et L3, (et M1 si dispo)'     : 'Moyenne',
    'Moyenne finale'                              : 'Moyenne_finale',
    'niveau en math'                              : 'Niveau_Math',
    'Alternance'                                  : 'Alternance',
    'Licence EDN '                                : 'Licence_EDN',
    'Classement'                                  : 'Classement',
}

cols = {k: v for k, v in COLONNES.items() if k and k in df_raw.columns}
missing = [k for k, v in COLONNES.items() if k and k not in df_raw.columns]
if missing:
    print('Colonnes introuvables :', missing)
else:
    print('Toutes les colonnes trouvees.')

df = df_raw[list(cols.keys())].rename(columns=cols).copy()
df.head(5)

Toutes les colonnes trouvees.


,ID,Diplome,Moyenne,Moyenne_finale,Niveau_Math,Alternance,Licence_EDN,Classement
0,1,Licence math info,10.883333333333333,10.883333,sup à 12,0,NaN,NC
1,2,Licence info,13,13.000000,sup à 12\n,0,0,64
2,3,licence autre,10.24,10.240000,NaN,0,NaN,NC
3,4,BUT info,9.94,9.940000,inf à 12,0,0,NC
4,5,licence info,10.597,10.597000,inf à 12,0,0,NC


In [3]:
# Nettoyage général : espaces insécables, retours à la ligne, strip
for col in df.columns:
    if str(df[col].dtype) in ('object', 'string'):
        df[col] = (
            df[col].astype(str)
            .str.replace(chr(160), ' ', regex=False)
            .str.replace('\n', ' ', regex=False)
            .str.strip()
            .replace({'': np.nan, 'nan': np.nan})
        )

# Colonnes numériques directes
df['Moyenne']       = pd.to_numeric(df['Moyenne'],       errors='coerce')
df['Moyenne_finale']= pd.to_numeric(df['Moyenne_finale'], errors='coerce')
df['Alternance']    = pd.to_numeric(df['Alternance'],    errors='coerce')
df['Licence_EDN']   = pd.to_numeric(df['Licence_EDN'],  errors='coerce')

# Classement : valeurs numériques ou 'NC' (non classé)
# NC → rang max + 1 (pire position possible)
classement_num = pd.to_numeric(df['Classement'].replace('NC', np.nan), errors='coerce')
max_rang = classement_num.max()
df['Classement'] = df['Classement'].replace('NC', max_rang + 1)
df['Classement'] = pd.to_numeric(df['Classement'], errors='coerce')

print('Types après nettoyage :')
print(df.dtypes)
print(f'\nMoyenne_finale — non-null: {df["Moyenne_finale"].notna().sum()}/815')
print(f'Classement     — non-null: {df["Classement"].notna().sum()}/815')

Types après nettoyage :
ID                  int64
Diplome               str
Moyenne           float64
Moyenne_finale    float64
Niveau_Math           str
Alternance        float64
Licence_EDN       float64
Classement        float64
dtype: object

Moyenne_finale — non-null: 802/815
Classement     — non-null: 815/815


## Niveau_Math — standardisation

Les 8+ variantes textuelles sont des doublons d'une même information.
On les réduit à deux labels propres : `inf_12` / `sup_12`.
L'utilisateur assignera les scores dans l'interface MCDM (étape 3).

In [4]:
print('Valeurs brutes Niveau_Math :')
print(df['Niveau_Math'].value_counts(dropna=False).to_string())

Valeurs brutes Niveau_Math :
Niveau_Math
sup à 12                                              348
inf à 12                                              342
NaN                                                   103
inf à 12\n                                              9
sup à 12\n                                              2
sup à 12\n(notes approx.: alphabétique)                 2
sup à 12\nmoyenne estimée                               1
sup à 12\n(moyenne estimée, notes alphanumériques)      1
sup à 12 \n                                             1
sup à 12\n(moyenne estimée)                             1
sup à 12\n                                              1
sup à12                                                 1
                                                        1
inf  à 12                                               1
sup à 12                                                1


In [5]:
def standardise_niveau_math(val):
    if pd.isna(val):
        return np.nan
    v = str(val).lower()
    if 'inf' in v:
        return 'inf_12'
    if 'sup' in v:
        return 'sup_12'
    return np.nan

df['Niveau_Math'] = df['Niveau_Math'].apply(standardise_niveau_math)
print('Distribution :', df['Niveau_Math'].value_counts(dropna=False).to_dict())

Distribution : {'sup_12': 359, 'inf_12': 352, nan: 104}


## Diplome — conservé tel quel

Le contenu de la colonne Diplome est gardé sans transformation.
Seuls les espaces superflus et caractères d'encodage ont été nettoyés (étape 3).

In [6]:
print(f'Valeurs uniques Diplome : {df["Diplome"].nunique(dropna=True)}')
print(df['Diplome'].value_counts(dropna=False).head(20).to_string())

Valeurs uniques Diplome : 165
Diplome
licence info          149
BUT info               96
BUT info               56
Licence info           37
licence autre          36
master info            32
licence autre          30
Licence math info      21
licence math           20
licence info           17
master/ing info        13
master/ing info        12
BUT autre              12
master autre           11
BUT statistiques       10
master/ing autre       10
Bachelor info           9
master autre            9
licence maths           7
licence prof info       7


## Diplome_norm - normalisation des diplomes info/math

Pour chaque valeur contenant `info`/`informatique` et/ou `math`/`maths`,
on construit un label propre : {Type} informatique, {Type} mathematique ou {Type} informatique/mathematique.
Les autres valeurs sont conservees telles quelles.
La colonne `Diplome` originale est preservee pour verification.

In [7]:
import re

def normalise_diplome(val):
    if pd.isna(val) or str(val).strip() in ('', 'nan'):
        return np.nan
    clean = re.sub(r'[\n\r\xa0]+', ' ', str(val)).strip()
    clean = re.sub(r'\s+', ' ', clean)
    v = clean.lower()
    has_info = bool(re.search(r'\binfo\b|\binformatique\b|\binformatiques\b', v))
    has_math = bool(re.search(r'\bmath\b|\bmaths\b', v))
    if has_info and has_math:
        domain = 'informatique/mathematique'
    elif has_info:
        domain = 'informatique'
    elif has_math:
        domain = 'mathematique'
    else:
        return clean
    if re.match(r'licence\s*(pro|prof)', v):
        typ = 'Licence Pro'
    elif re.match(r'licence', v):
        typ = 'Licence'
    elif re.match(r'master/ing|maste/ing', v):
        typ = 'Master/Ingenieur'
    elif re.match(r'mast', v):
        typ = 'Master'
    elif re.match(r'but\b', v):
        typ = 'BUT'
    elif re.match(r'bts\b', v):
        typ = 'BTS'
    elif re.match(r'dut\b', v):
        typ = 'DUT'
    elif re.match(r'bachelor', v):
        typ = 'Bachelor'
    elif re.match(r'ing|.cole\s*ing|cycle\s*ing', v):
        typ = 'Ingenieur'
    elif re.match(r'g.nie', v):
        typ = 'Genie'
    elif re.match(r'autre', v):
        typ = 'Autre'
    else:
        first = clean.split()[0] if clean.split() else 'Autre'
        typ = first.capitalize()
    return f'{typ} {domain}'

df['Diplome_norm'] = df['Diplome'].apply(normalise_diplome)
print(f"Valeurs uniques Diplome_norm : {df['Diplome_norm'].nunique(dropna=True)}")
print(df['Diplome_norm'].value_counts(dropna=False).to_string())

Valeurs uniques Diplome_norm : 110
Diplome_norm
Licence informatique                                                  205
BUT informatique                                                      155
licence autre                                                          66
Master informatique                                                    51
Licence mathematique                                                   39
Master/Ingenieur informatique                                          36
Licence informatique/mathematique                                      35
Bachelor informatique                                                  21
master autre                                                           20
master/ing autre                                                       13
BUT autre                                                              12
BUT statistiques                                                       10
Licence Pro informatique                                        

In [8]:
# Verification : Diplome original vs Diplome_norm (lignes transformees uniquement)
mask = df['Diplome'].fillna('') != df['Diplome_norm'].fillna('')
comp = df.loc[mask, ['Diplome', 'Diplome_norm']].drop_duplicates().sort_values('Diplome_norm')
print(f"{len(comp)} valeurs transformees:\n")
pd.set_option('display.max_colwidth', None)
print(comp.to_string(index=False))

81 valeurs transformees:

                                                                        Diplome                                                       Diplome_norm
                     Autre (info, pro, validation de compétences, pas de notes)                                                 Autre informatique
          Autre info\n (titre professionnel Développeur web et mobile, niveau 5                                                 Autre informatique
                                                          Autre (info, MSc pro)                                                 Autre informatique
                                                                           BTS                                                                 BTS
                                                      BTS  info + Bachelor info                                                   BTS informatique
                      BTS non info + autre diplôme (type inconnu?) data analyst             

In [9]:
# Dataset final - Diplome_norm inclus pour verification
df_final = df[['ID', 'Diplome', 'Diplome_norm', 'Moyenne', 'Moyenne_finale',
               'Niveau_Math', 'Alternance', 'Licence_EDN', 'Classement']].copy()

print('Types :')
print(df_final.dtypes)
print('\nValeurs manquantes :')
print(df_final.isna().sum())
df_final.head()

Types :
ID                  int64
Diplome               str
Diplome_norm          str
Moyenne           float64
Moyenne_finale    float64
Niveau_Math           str
Alternance        float64
Licence_EDN       float64
Classement        float64
dtype: object

Valeurs manquantes :
ID                  0
Diplome             0
Diplome_norm        0
Moyenne            29
Moyenne_finale     13
Niveau_Math       104
Alternance         23
Licence_EDN       311
Classement          0
dtype: int64


,ID,Diplome,Diplome_norm,Moyenne,Moyenne_finale,Niveau_Math,Alternance,Licence_EDN,Classement
0,1,Licence math info,Licence informatique/mathematique,10.883333,10.883333,sup_12,0.0,NaN,422.0
1,2,Licence info,Licence informatique,13.000000,13.000000,sup_12,0.0,0.0,64.0
2,3,licence autre,licence autre,10.240000,10.240000,NaN,0.0,NaN,422.0
3,4,BUT info,BUT informatique,9.940000,9.940000,inf_12,0.0,0.0,422.0
4,5,licence info,Licence informatique,10.597000,10.597000,inf_12,0.0,0.0,422.0


## Création de la colonne Tier (cible ML)

Les tiers sont assignés par **quartiles de `Moyenne_finale`** (approche bootstrap du système) :

| Quartile | Tier |
|---|---|
| ≥ Q75 | **Excellent** |
| ≥ Q50 | **Bon** |
| ≥ Q25 | **Moyen** |
| < Q25 | **Faible** |

On exclut d'abord les lignes sans `Moyenne_finale` (13 lignes) pour calculer les quartiles sur données réelles.

In [10]:
q25, q50, q75 = df_final['Moyenne_finale'].quantile([0.25, 0.50, 0.75])
print(f"Quartiles Moyenne_finale :")
print(f"  Q25 = {q25:.3f}  →  < {q25:.2f}       : Faible")
print(f"  Q50 = {q50:.3f}  →  [{q25:.2f}, {q50:.2f}[ : Moyen")
print(f"  Q75 = {q75:.3f}  →  [{q50:.2f}, {q75:.2f}[ : Bon")
print(f"                →  ≥ {q75:.2f}       : Excellent")

def assign_tier(val):
    if pd.isna(val):
        return np.nan
    if val >= q75: return "Excellent"
    if val >= q50: return "Bon"
    if val >= q25: return "Moyen"
    return "Faible"

df_final['Tier'] = df_final['Moyenne_finale'].apply(assign_tier)

print("\nDistribution des tiers :")
print(df_final['Tier'].value_counts(dropna=False).to_string())

df_final[['ID', 'Moyenne_finale', 'Tier']].head(10)

Quartiles Moyenne_finale :
  Q25 = 11.019  →  < 11.02       : Faible
  Q50 = 12.142  →  [11.02, 12.14[ : Moyen
  Q75 = 13.351  →  [12.14, 13.35[ : Bon
                →  ≥ 13.35       : Excellent

Distribution des tiers :
Tier
Faible       201
Excellent    201
Bon          200
Moyen        200
NaN           13


,ID,Moyenne_finale,Tier
0,1,10.883333,Faible
1,2,13.000000,Bon
2,3,10.240000,Faible
3,4,9.940000,Faible
4,5,10.597000,Faible
5,6,10.400000,Faible
6,7,12.060000,Moyen
7,8,10.380000,Faible
8,9,11.000000,Faible
9,10,12.120000,Moyen


In [ ]:
OUTPUT = 'candidats_mcdm.csv'
df_final.to_csv(OUTPUT, index=False, encoding='utf-8')
print(f'Fichier exporte : {OUTPUT}')
print(f'{df_final.shape[0]} candidats, colonnes : {list(df_final.columns)}')
df_final.head()

Fichier exporte : candidats_mcdm.csv
815 candidats, colonnes : ['ID', 'Diplome', 'Diplome_norm', 'Moyenne', 'Moyenne_finale', 'Niveau_Math', 'Alternance', 'Licence_EDN', 'Classement', 'Tier']


,ID,Diplome,Diplome_norm,Moyenne,Moyenne_finale,Niveau_Math,Alternance,Licence_EDN,Classement,Tier
0,1,Licence math info,Licence informatique/mathematique,10.883333,10.883333,sup_12,0.0,NaN,422.0,Faible
1,2,Licence info,Licence informatique,13.000000,13.000000,sup_12,0.0,0.0,64.0,Bon
2,3,licence autre,licence autre,10.240000,10.240000,NaN,0.0,NaN,422.0,Faible
3,4,BUT info,BUT informatique,9.940000,9.940000,inf_12,0.0,0.0,422.0,Faible
4,5,licence info,Licence informatique,10.597000,10.597000,inf_12,0.0,0.0,422.0,Faible


: 